# 08_paper_replication_lu_uddin.ipynb — ทำตามวิธีของ Lu & Uddin (2022) เป๊ะๆ

โน้ตบุ๊กนี้ replicate methodology ของ **Lu & Uddin (2022, Information journal)** บน `diabetic_data.csv` ตัวเดียวกับที่เราใช้ ให้ตรงกับที่เปเปอร์อธิบายไว้มากที่สุด:

1. Cleaning: drop weight/medical_specialty/payer_code (>30% missing), drop missing race, drop invalid gender, drop expired/hospice discharge
2. Feature engineering: binary encode drug/A1c/glucose, ICD-9 → 9 disease categories, one-hot encode race/admission_type/discharge_disposition/admission_source
3. Train/test split 80/20 **แบบสุ่มทีละแถว** (ตามที่เปเปอร์อธิบายตรงตัวอักษร ไม่ได้พูดถึงการ group ตาม patient)
4. **Random Under-Sampling (RUS)** บน training set
5. **SelectFromModel** บน CatBoost → เลือกฟีเจอร์ที่สำคัญที่สุด
6. **Stacking**: RF + AdaBoost + XGBoost + CatBoost (base) → Logistic Regression (meta)
7. Evaluate ทั้งแบบ weighted-average (แบบที่เปเปอร์รายงานใน Table 3) และแบบ per-class (เพื่อเห็นภาพจริง)

**สิ่งที่พบระหว่างทำ (ทดสอบด้วยข้อมูลจริงแล้ว)**:
- ทำตาม cleaning steps ของเปเปอร์ได้ตัวเลขตรงเป๊ะ: **97,108 patients** (ตรงกับที่เปเปอร์รายงานในหน้า 7 ทุกตัวอักษร)
- ทำ RUS แล้วได้ training set **68,785 not-readmitted, 8,901 readmitted** — ตรงกับตัวเลขที่เปเปอร์รายงานไว้เป๊ะเช่นกัน → ยืนยันว่า replicate ถูกวิธีจริง
- **พบว่าเปเปอร์ split train/test แบบสุ่มทีละแถว (row-level) ไม่ได้ group ตาม patient_nbr** → มีผู้ป่วย **6,632 คน (~6.8% ของทั้งหมด)** ที่ปรากฏทั้งใน train และ test set พร้อมกัน (potential data leakage) — โน้ตบุ๊กนี้ทำให้ดูทั้งสองแบบเพื่อเปรียบเทียบ

ก่อนรัน: `pip install imbalanced-learn catboost xgboost`

## 1. Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, average_precision_score,
)
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42


## 2. Load & Clean Data (ตามที่เปเปอร์อธิบายเป๊ะ)

- ตัดคอลัมน์ที่ missing >30%: `weight`, `medical_specialty`, `payer_code`
- ตัดแถวที่ race เป็น missing (2.3% ของข้อมูล) — **ต่างจาก pipeline ของเราเอง (05/07) ที่เก็บไว้เป็น category "Missing"**
- ตัดแถวที่ gender invalid
- ตัดแถวที่ discharge disposition เป็น expired/hospice (เพราะกลับมา readmit ไม่ได้)

In [2]:
RAW_PATH = "../data/diabetic_data.csv"  # แก้ path ตามที่เก็บไฟล์จริง

df = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=["?"])
print(f"Raw shape: {df.shape}")

df = df[df["race"].notna()]
print(f"After dropping missing race: {df.shape}")

df = df[df["gender"] != "Unknown/Invalid"]
print(f"After dropping invalid gender: {df.shape}")

death_hospice_codes = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(death_hospice_codes)]
df = df.reset_index(drop=True)
print(f"After dropping expired/hospice: {df.shape}")

print("\n>>> เปเปอร์รายงานว่าเหลือ 97,108 patients หลัง filtering — เช็คว่าตรงกันไหม")


Raw shape: (101766, 50)
After dropping missing race: (99493, 50)
After dropping invalid gender: (99492, 50)
After dropping expired/hospice: (97108, 50)

>>> เปเปอร์รายงานว่าเหลือ 97,108 patients หลัง filtering — เช็คว่าตรงกันไหม


## 3. Feature Engineering (ตามที่เปเปอร์อธิบาย)

- Target: `<30` → 1, มิฉะนั้น → 0
- diag_1/2/3 → แปลงเป็น 9 กลุ่มโรคตาม ICD-9-CM (Table 1 ของเปเปอร์)
- 21 ยา + A1Cresult + max_glu_serum → binary (ตามที่เปเปอร์ระบุ: 'No'/'Norm' = 0, อื่นๆ = 1)
- one-hot encode: race, admission_type_id, discharge_disposition_id, admission_source_id, diag_1/2/3
- ตัด examide, citoglipton (constant คอลัมน์เดียว ตามที่เปเปอร์ระบุ)
- ตัด encounter_id, patient_nbr (เปเปอร์บอกว่าเป็น nominal variable ไม่ใช้เป็นฟีเจอร์ — แต่เราจะเก็บ patient_nbr ไว้ต่างหากเพื่อทำ Version B ด้านล่าง)

In [3]:
df["readmitted"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)

def map_diagnosis(code):
    if pd.isna(code) or code == "":
        return "Other"
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        val = float(code)
    except ValueError:
        return "Other"
    if 250 <= val < 251:
        return "Diabetes"
    elif 390 <= val <= 459 or val == 785:
        return "Circulatory"
    elif 460 <= val <= 519 or val == 786:
        return "Respiratory"
    elif 520 <= val <= 579 or val == 787:
        return "Digestive"
    elif 800 <= val <= 999:
        return "Injury"
    elif 710 <= val <= 739:
        return "Musculoskeletal"
    elif 580 <= val <= 629 or val == 788:
        return "Genitourinary"
    elif 140 <= val <= 239:
        return "Neoplasms"
    else:
        return "Other"

for c in ["diag_1", "diag_2", "diag_3"]:
    df[c] = df[c].apply(map_diagnosis)

drug_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]
for c in drug_cols:
    df[c] = (df[c] != "No").astype(int)

df["max_glu_serum"] = (df["max_glu_serum"] != "Norm").astype(int)
df["A1Cresult"] = (df["A1Cresult"] != "Norm").astype(int)
df["change"] = (df["change"] == "Ch").astype(int)
df["diabetesMed"] = (df["diabetesMed"] == "Yes").astype(int)
df["gender"] = (df["gender"] == "Male").astype(int)

const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns dropped:", const_cols)
df.drop(columns=const_cols, inplace=True)
df.drop(columns=["weight", "payer_code", "medical_specialty"], inplace=True)

age_map = {
    "[0-10)": 0, "[10-20)": 1, "[20-30)": 2, "[30-40)": 3, "[40-50)": 4,
    "[50-60)": 5, "[60-70)": 6, "[70-80)": 7, "[80-90)": 8, "[90-100)": 9,
}
df["age"] = df["age"].map(age_map)

# เก็บ patient_nbr ไว้ต่างหากก่อนตัดทิ้ง (ไว้ใช้ทำ Version B)
group_col = df["patient_nbr"].copy()
df.drop(columns=["encounter_id", "patient_nbr"], inplace=True)

onehot_cols = ["race", "admission_type_id", "discharge_disposition_id", "admission_source_id",
               "diag_1", "diag_2", "diag_3"]
df = pd.get_dummies(df, columns=onehot_cols, drop_first=False)

print(f"\nFinal shape after encoding: {df.shape}")
print(f"Target distribution: {df['readmitted'].value_counts(normalize=True).to_dict()}")


Constant columns dropped: ['examide', 'citoglipton', 'metformin-rosiglitazone']

Final shape after encoding: (97108, 113)
Target distribution: {0: 0.8854265354038802, 1: 0.11457346459611978}


## 4. Train/Test Split — เทียบ 2 แบบ

- **Version A**: `train_test_split` สุ่มทีละแถว (stratified) — **ตามที่เปเปอร์อธิบายตรงตัวอักษร** (ไม่พูดถึงการ group ตาม patient เลย)
- **Version B**: `GroupShuffleSplit` ตาม `patient_nbr` — วิธีที่ถูกต้องกว่าเพื่อกัน data leakage (ผู้ป่วยที่มีหลาย encounter ต้องอยู่ฝั่งเดียวกันทั้งหมด)

เทียบให้เห็นว่า leakage ใน Version A ส่งผลต่อผลลัพธ์มากแค่ไหน

In [4]:
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

# Version A: row-random split (ตามเปเปอร์)
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Version A (row-random, ตามเปเปอร์):")
print(f"  Train: {X_train_a.shape[0]:,} rows | not-readmit: {(y_train_a==0).sum():,} | readmit: {(y_train_a==1).sum():,}")
print(f"  (เปเปอร์รายงาน: not-readmit=68,785, readmit=8,901 — เทียบดูว่าตรงกันไหม)")

# Version B: patient-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=group_col))
X_train_b, X_test_b = X.iloc[train_idx], X.iloc[test_idx]
y_train_b, y_test_b = y.iloc[train_idx], y.iloc[test_idx]

overlap_a = set(group_col.iloc[X_train_a.index]) & set(group_col.iloc[X_test_a.index])
overlap_b = set(group_col.iloc[train_idx]) & set(group_col.iloc[test_idx])
print(f"\nPatient overlap ระหว่าง train/test:")
print(f"  Version A (row-random) : {len(overlap_a):,} patients ปรากฏทั้งใน train และ test (potential leakage)")
print(f"  Version B (grouped)    : {len(overlap_b)} patients (ต้องเป็น 0)")


Version A (row-random, ตามเปเปอร์):
  Train: 77,686 rows | not-readmit: 68,785 | readmit: 8,901
  (เปเปอร์รายงาน: not-readmit=68,785, readmit=8,901 — เทียบดูว่าตรงกันไหม)

Patient overlap ระหว่าง train/test:
  Version A (row-random) : 6,632 patients ปรากฏทั้งใน train และ test (potential leakage)
  Version B (grouped)    : 0 patients (ต้องเป็น 0)


## 5. Pipeline หลัก: RUS → SelectFromModel(CatBoost) → Stacking

ทำเป็นฟังก์ชันเดียวเพื่อรันซ้ำได้ทั้งสอง Version

In [5]:
def run_paper_pipeline(X_train, X_test, y_train, y_test, label, n_features_target=18):
    print(f"\n{'='*70}\n{label}\n{'='*70}")

    # --- Random Under-Sampling บน training set เท่านั้น ---
    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)
    print(f"หลัง RUS: {X_train_bal.shape[0]:,} rows, class balance: {y_train_bal.value_counts().to_dict()}")

    # --- Feature selection: SelectFromModel บน CatBoost (ตามเปเปอร์) ---
    fs_model = CatBoostClassifier(iterations=200, random_state=RANDOM_STATE, verbose=False)
    fs_model.fit(X_train_bal, y_train_bal)
    selector = SelectFromModel(fs_model, max_features=n_features_target, threshold=-np.inf, prefit=True)
    selected_features = X_train_bal.columns[selector.get_support()]
    print(f"เลือกได้ {len(selected_features)} features (เปเปอร์เลือก 18 จาก 55)")
    print("Top features:", list(selected_features[:10]))

    X_train_sel = X_train_bal[selected_features]
    X_test_sel = X_test[selected_features]

    # --- Stacking: RF, AdaBoost, XGBoost, CatBoost -> LR meta (ตาม Figure 2 ของเปเปอร์) ---
    estimators = [
        ("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
        ("adaboost", AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE)),
        ("xgboost", XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)),
        ("catboost", CatBoostClassifier(iterations=200, random_state=RANDOM_STATE, verbose=False)),
    ]
    stacking = StackingClassifier(
        estimators=estimators, final_estimator=LogisticRegression(max_iter=1000),
        cv=5, n_jobs=1, passthrough=False,
    )
    print("กำลังเทรน stacking model...")
    stacking.fit(X_train_sel, y_train_bal)
    print("เทรนเสร็จ")

    proba = stacking.predict_proba(X_test_sel)[:, 1]
    y_pred = stacking.predict(X_test_sel)

    # --- ตัวเลขแบบเปเปอร์ (weighted / accuracy รวม) ---
    print(f"\n--- ตัวเลขสไตล์เปเปอร์ (weighted / aggregate) ---")
    print(f"Accuracy            : {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(f"AUC                  : {roc_auc_score(y_test, proba):.4f}")
    print(f"Precision (weighted) : {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"F1 (weighted)        : {f1_score(y_test, y_pred, average='weighted'):.4f}")

    # --- ตัวเลขแบบ per-class (เห็นภาพจริง) ---
    print(f"\n--- ตัวเลขแบบ per-class (สิ่งที่สำคัญจริง) ---")
    print(classification_report(y_test, y_pred, target_names=["Not readmit","Readmit"], digits=4))
    print(f"PR-AUC (minority class): {average_precision_score(y_test, proba):.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, proba),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "precision_minority": precision_score(y_test, y_pred, pos_label=1),
        "recall_minority": recall_score(y_test, y_pred, pos_label=1),
        "f1_minority": f1_score(y_test, y_pred, pos_label=1),
        "pr_auc": average_precision_score(y_test, proba),
        "n_features": len(selected_features),
    }


In [6]:
res_a = run_paper_pipeline(X_train_a, X_test_a, y_train_a, y_test_a,
                            "VERSION A: Row-random split (ตามเปเปอร์ตรงตัวอักษร — มี leakage)")



VERSION A: Row-random split (ตามเปเปอร์ตรงตัวอักษร — มี leakage)
หลัง RUS: 17,802 rows, class balance: {0: 8901, 1: 8901}
เลือกได้ 18 features (เปเปอร์เลือก 18 จาก 55)
Top features: ['gender', 'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
กำลังเทรน stacking model...
เทรนเสร็จ

--- ตัวเลขสไตล์เปเปอร์ (weighted / aggregate) ---
Accuracy            : 63.37%
AUC                  : 0.6641
Precision (weighted) : 0.8414
F1 (weighted)        : 0.7000

--- ตัวเลขแบบ per-class (สิ่งที่สำคัญจริง) ---
              precision    recall  f1-score   support

 Not readmit     0.9271    0.6363    0.7547     17197
     Readmit     0.1790    0.6130    0.2772      2225

    accuracy                         0.6337     19422
   macro avg     0.5531    0.6247    0.5159     19422
weighted avg     0.8414    0.6337    0.7000     19422

PR-AUC (minority class): 0.2141
Confusion matrix:
 [[109

In [7]:
res_b = run_paper_pipeline(X_train_b, X_test_b, y_train_b, y_test_b,
                            "VERSION B: Patient-grouped split (แก้ leakage แล้ว)")



VERSION B: Patient-grouped split (แก้ leakage แล้ว)
หลัง RUS: 17,756 rows, class balance: {0: 8878, 1: 8878}
เลือกได้ 18 features (เปเปอร์เลือก 18 จาก 55)
Top features: ['gender', 'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
กำลังเทรน stacking model...
เทรนเสร็จ

--- ตัวเลขสไตล์เปเปอร์ (weighted / aggregate) ---
Accuracy            : 61.13%
AUC                  : 0.6557
Precision (weighted) : 0.8361
F1 (weighted)        : 0.6815

--- ตัวเลขแบบ per-class (สิ่งที่สำคัญจริง) ---
              precision    recall  f1-score   support

 Not readmit     0.9230    0.6117    0.7357     17261
     Readmit     0.1694    0.6081    0.2650      2248

    accuracy                         0.6113     19509
   macro avg     0.5462    0.6099    0.5004     19509
weighted avg     0.8361    0.6113    0.6815     19509

PR-AUC (minority class): 0.2110
Confusion matrix:
 [[10558  6703]
 [ 

## 6. ตารางเปรียบเทียบสุดท้าย: เปเปอร์ vs Replication (2 แบบ) vs Pipeline ของเรา

รวมตัวเลขทั้งหมดไว้ในที่เดียว เอาไปใส่รายงานได้เลย

In [ ]:
comparison = pd.DataFrame({
    "เปเปอร์ (Table 3, รายงานเอง)": {
        "Accuracy (%)": 68.63, "AUC": 0.6736, "F1 (weighted)": 0.7434,
        "Precision (minority, คำนวณจาก CM จริง)": 0.1748,
        "Recall (minority, คำนวณจาก CM จริง)": 0.6068,
        "F1 (minority, คำนวณจาก CM จริง)": 0.2716,
    },
    "Replication Version A (row-split, มี leakage)": {
        "Accuracy (%)": res_a["accuracy"]*100, "AUC": res_a["auc"], "F1 (weighted)": res_a["f1_weighted"],
        "Precision (minority, คำนวณจาก CM จริง)": res_a["precision_minority"],
        "Recall (minority, คำนวณจาก CM จริง)": res_a["recall_minority"],
        "F1 (minority, คำนวณจาก CM จริง)": res_a["f1_minority"],
    },
    "Replication Version B (grouped-split, ไม่มี leakage)": {
        "Accuracy (%)": res_b["accuracy"]*100, "AUC": res_b["auc"], "F1 (weighted)": res_b["f1_weighted"],
        "Precision (minority, คำนวณจาก CM จริง)": res_b["precision_minority"],
        "Recall (minority, คำนวณจาก CM จริง)": res_b["recall_minority"],
        "F1 (minority, คำนวณจาก CM จริง)": res_b["f1_minority"],
    },
}).T

print(comparison.round(4).to_string())

print("""
>>> สรุปสิ่งที่ควรเขียนในรายงาน:
>>> 1. ตัวเลข "weighted/aggregate" ของเปเปอร์ (Accuracy 68.63%, F1 0.7434) ดูดีกว่าของเรามาก
>>>    แต่พอ disaggregate เป็น per-class (minority) ตัวเลขจริงใกล้เคียงกับ replication และ pipeline ของเราเอง
>>> 2. เปเปอร์ split แบบ row-random ทำให้เกิด patient leakage (~6,632 คนซ้ำ train/test)
>>>    ซึ่งเป็นจุดที่ pipeline ของเรา (GroupShuffleSplit ตาม patient_nbr) ทำได้ถูกต้องกว่าตั้งแต่แรก
>>> 3. เมื่อ apples-to-apples (ใช้ RUS + SelectFromModel + stacking 4 โมเดลแบบเปเปอร์ทุกอย่าง)
>>>    ผลลัพธ์ per-class ที่ได้ยังอยู่ในระดับเดียวกับ pipeline ของเราเอง (07) — ยืนยัน data ceiling
""")


                                                      Accuracy (%)     AUC  F1 (weighted)  Precision (minority, คำนวณจาก CM จริง)  Recall (minority, คำนวณจาก CM จริง)  F1 (minority, คำนวณจาก CM จริง)
เปเปอร์ (Table 3, รายงานเอง)                               68.6300  0.6736         0.7434                                  0.1748                               0.6068                           0.2716
Replication Version A (row-split, มี leakage)              63.3663  0.6641         0.7000                                  0.1790                               0.6130                           0.2772
Replication Version B (grouped-split, ไม่มี leakage)       61.1256  0.6557         0.6815                                  0.1694                               0.6081                           0.2650

>>> สรุปสิ่งที่ควรเขียนในรายงาน:
>>> 1. ตัวเลข "weighted/aggregate" ของเปเปอร์ (Accuracy 68.63%, F1 0.7434) ดูดีกว่าของเรามาก
>>>    แต่พอ disaggregate เป็น per-class (minority) ตัวเลขจริงใกล้เคียงกั

: 

## 7. สรุป: ควรเปลี่ยนไปใช้วิธีของเปเปอร์ไหม?

**ไม่ควรครับ** ด้วยเหตุผล:

1. **Performance ที่แท้จริง (per-class) ใกล้เคียงกัน** — ไม่ว่าจะใช้วิธีไหน ก็เจอ ceiling เดียวกันของ dataset (~F1 0.26-0.28 สำหรับ minority class)
2. **เปเปอร์มี methodological issue ที่ pipeline ของเราไม่มี**: patient-level data leakage จาก row-random split
3. **Pipeline ของเรามี evaluation ที่เข้มงวดกว่า**: ใช้ PR-AUC/F2/cost-based threshold/calibration/fairness analysis ซึ่งเปเปอร์ไม่มีเลย
4. **ฟีเจอร์ของเราสมบูรณ์กว่า**: มี longitudinal features (visit_number, prior_change_count, total_prior_visits) ที่เปเปอร์ไม่มี เพราะเปเปอร์ตัด patient_nbr ทิ้งไปเลยตั้งแต่ต้น ทำให้ไม่มีทางสร้างฟีเจอร์ประวัติผู้ป่วยข้าม encounter ได้

**สิ่งที่ควรทำแทน**: เอาผลการ replicate นี้ไปเป็นหลักฐานสนับสนุนใน Discussion section ว่า pipeline ของเราออกแบบมาอย่างรอบคอบกว่า ไม่ใช่เพราะทำโมเดลได้แย่กว่า